In [3]:
import pandas as pd
import numpy as np

print(pd.__version__)
print(np.__version__)

3.0.2
2.4.4


In [4]:
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

In [5]:
num_users = 10000

user_ids = range(1, num_users + 1)

user_types = np.random.choice(
    ['student', 'professional'],
    size=num_users,
    p=[0.6, 0.4]
)

experience_levels = np.random.choice(
    ['beginner', 'intermediate', 'advanced'],
    size=num_users,
    p=[0.5, 0.3, 0.2]
)

start_date = datetime(2024, 1, 1)

signup_dates = [
    start_date + timedelta(days=random.randint(0, 90))
    for _ in range(num_users)
]

users_df = pd.DataFrame({
    'user_id': user_ids,
    'user_type': user_types,
    'experience_level': experience_levels,
    'signup_date': signup_dates
})

users_df.head()

,user_id,user_type,experience_level,signup_date
0,1,student,beginner,2024-03-22
1,2,professional,beginner,2024-01-15
2,3,professional,beginner,2024-01-04
3,4,student,intermediate,2024-02-05
4,5,student,beginner,2024-02-01


In [6]:
sessions = []
session_id = 1

for _, user in users_df.iterrows():
    num_sessions = random.randint(1, 3)
    
    for s in range(num_sessions):
        sessions.append({
            'session_id': session_id,
            'user_id': user['user_id'],
            'session_number': s + 1,
            'session_date': user['signup_date'] + timedelta(days=s)
        })
        
        session_id += 1

sessions_df = pd.DataFrame(sessions)
sessions_df.head()

,session_id,user_id,session_number,session_date
0,1,1,1,2024-03-22
1,2,1,2,2024-03-23
2,3,2,1,2024-01-15
3,4,3,1,2024-01-04
4,5,3,2,2024-01-05


In [7]:
sessions_df.shape

(19935, 4)

In [8]:
prompts = []
prompt_id = 1

for _, session in sessions_df.iterrows():
    num_prompts = random.randint(1, 5)
    
    user = users_df[users_df['user_id'] == session['user_id']].iloc[0]
    
    for attempt in range(num_prompts):
        
        # Prompt type logic
        if user['user_type'] == 'student':
            prompt_type = np.random.choice(['assignment', 'summary'], p=[0.7, 0.3])
        else:
            prompt_type = np.random.choice(['report', 'summary'], p=[0.7, 0.3])
        
        # Prompt quality logic
        if user['experience_level'] == 'beginner':
            prompt_quality = random.randint(3, 5)
        elif user['experience_level'] == 'intermediate':
            prompt_quality = random.randint(5, 7)
        else:
            prompt_quality = random.randint(7, 9)
        
        prompts.append({
            'prompt_id': prompt_id,
            'session_id': session['session_id'],
            'user_id': session['user_id'],
            'prompt_type': prompt_type,
            'prompt_quality_score': prompt_quality,
            'prompt_attempt_number': attempt + 1
        })
        
        prompt_id += 1

prompts_df = pd.DataFrame(prompts)
prompts_df.head()

,prompt_id,session_id,user_id,prompt_type,prompt_quality_score,prompt_attempt_number
0,1,1,1,summary,4,1
1,2,1,1,assignment,4,2
2,3,2,1,assignment,4,1
3,4,2,1,assignment,3,2
4,5,2,1,assignment,3,3


In [9]:
prompts_df.shape

(59842, 6)

In [10]:
prompts_df['prompt_type'].value_counts()

prompt_type
assignment    25517
summary       17903
report        16422
Name: count, dtype: int64

In [11]:
prompts_df['prompt_quality_score'].describe()

count    59842.000000
mean         5.414859
std          1.764055
min          3.000000
25%          4.000000
50%          5.000000
75%          7.000000
max          9.000000
Name: prompt_quality_score, dtype: float64

In [12]:
outputs = []
output_id = 1

for _, prompt in prompts_df.iterrows():
    
    base = prompt['prompt_quality_score']
    noise = random.randint(-1, 2)
    
    output_quality = max(1, min(10, base + noise))
    
    feedback = 'good' if output_quality >= 7 else 'bad'
    
    edited = 1 if output_quality < 7 else 0
    
    good_flag = 1 if output_quality >= 7 else 0
    
    outputs.append({
        'output_id': output_id,
        'prompt_id': prompt['prompt_id'],
        'output_quality_score': output_quality,
        'feedback': feedback,
        'edited_output': edited,
        'good_output_flag': good_flag
    })
    
    output_id += 1

outputs_df = pd.DataFrame(outputs)
outputs_df.head()

,output_id,prompt_id,output_quality_score,feedback,edited_output,good_output_flag
0,1,1,5,bad,1,0
1,2,2,5,bad,1,0
2,3,3,5,bad,1,0
3,4,4,4,bad,1,0
4,5,5,3,bad,1,0


In [13]:
outputs_df.shape

(59842, 6)

In [14]:
outputs_df['feedback'].value_counts()

feedback
bad     37176
good    22666
Name: count, dtype: int64

In [15]:
# Merge outputs with prompts to get user_id
merged_df = outputs_df.merge(prompts_df, on='prompt_id')

# Find users with at least one good output
users_with_good_output = merged_df[
    merged_df['good_output_flag'] == 1
]['user_id'].unique()

# Add retention flag
users_df['retained'] = users_df['user_id'].apply(
    lambda x: 1 if x in users_with_good_output else 0
)

users_df.head()

,user_id,user_type,experience_level,signup_date,retained
0,1,student,beginner,2024-03-22,0
1,2,professional,beginner,2024-01-15,1
2,3,professional,beginner,2024-01-04,0
3,4,student,intermediate,2024-02-05,1
4,5,student,beginner,2024-02-01,0


In [16]:
users_df['retained'].value_counts()

retained
1    6760
0    3240
Name: count, dtype: int64